# Notebook 03: Baseline Convolutional Neural Network (SimpleCNN)

**Objective:** Build, understand, train, and evaluate a baseline Convolutional Neural Network from scratch for industrial defect classification.

### Core Concepts Explored:
1. **Convolution Mathematics:** 2D convolution filters, kernel sizes, strides, and padding.
2. **Spatial Feature Maps:** How intermediate feature representations evolve from low-level edges to semantic defect patterns.
3. **Global Average Pooling (GAP) vs Flatten:** Why GAP prevents parameter explosion and overfitting.
4. **PyTorch Training Loop:** Forward pass, loss computation, backpropagation, and optimizer updates.

In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.models.classification.simple_cnn import SimpleCNN
from src.evaluation.classification_metrics import compute_classification_metrics, plot_confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Mathematical Architecture Breakdown

For a 2D convolution with input size $H_{\text{in}} \times W_{\text{in}}$, kernel $K$, padding $P$, and stride $S$:
$$H_{\text{out}} = \left\lfloor \frac{H_{\text{in}} + 2P - K}{S} + 1 \right\rfloor$$

With $K=3, P=1, S=1$:
$$H_{\text{out}} = \frac{H_{\text{in}} + 2(1) - 3}{1} + 1 = H_{\text{in}}$$
Spatial dimensions are preserved by the convolution and halved by each subsequent MaxPool2D ($2 \times 2$, stride $2$).

In [ ]:
model = SimpleCNN(num_classes=6, in_channels=3, dropout=0.3).to(device)
print(model)

params = model.get_num_parameters()
print(f"\nTotal Parameters:     {params['total_parameters']:,}")
print(f"Trainable Parameters: {params['trainable_parameters']:,}")

## 2. Forward Pass Tensor Shape Tracking

Let's trace a dummy input tensor $(B=2, C=3, H=224, W=224)$ through each sequential block.

In [ ]:
dummy_input = torch.randn(2, 3, 224, 224).to(device)

x = dummy_input
print(f"Input shape:                   {x.shape}")

for i, block in enumerate(model.features):
    x = block(x)
    print(f"Stage {i+1} Feature Map shape: {x.shape}")

x = model.global_pool(x)
print(f"Global Average Pool shape:     {x.shape}")

logits = model.classifier(x)
print(f"Output Logits shape:           {logits.shape}")

## 3. Visualizing Convolutional Feature Maps

Let's pass an image through the first convolutional layer and visualize the 32 learned feature maps (detecting oriented edges, color gradients, and defect textures).

In [ ]:
with torch.no_grad():
    first_block = model.features[0]
    feature_maps = first_block(dummy_input[:1])  # (1, 32, 112, 112)

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for idx, ax in enumerate(axes.flat):
    ax.imshow(feature_maps[0, idx].cpu().numpy(), cmap="viridis")
    ax.axis("off")
    ax.set_title(f"Ch {idx}", fontsize=8)

plt.suptitle("SimpleCNN Stage 1: 32 Convolutional Feature Maps", fontsize=13)
plt.tight_layout()
plt.show()